# AeroSense — Kaggle run: explore → prepare → train

Run cells top to bottom. **Before starting:** Settings → Accelerator = **GPU T4**, Internet = **On**,
and both datasets attached (`orvile/coughvid-v3`, `vbookshelf/respiratory-sound-database`).

## 0. Get the code and check the environment

In [ ]:
import os, sys, shutil, subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/AeroSense.git"   # <-- CHANGE THIS
REPO = "/kaggle/working/AeroSense"

if not os.path.exists(REPO):
    r = subprocess.run(["git", "clone", REPO_URL, REPO], capture_output=True, text=True)
    print(r.stdout or r.stderr)
assert os.path.exists(f"{REPO}/src/config.py"), (
    "Code not found. Fix REPO_URL (repo must be public), or upload your src/ folder "
    "so that /kaggle/working/AeroSense/src/config.py exists.")

os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")

import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print("ffmpeg:", shutil.which("ffmpeg"))
from config import ON_KAGGLE, COUGHVID_AUDIO_DIR, ICBHI_AUDIO_DIR
print("on kaggle:", ON_KAGGLE, "\ncoughvid audio dir exists:", COUGHVID_AUDIO_DIR.exists(),
      "\nicbhi audio dir exists:", ICBHI_AUDIO_DIR.exists())

## 1. Build the manifests (labels → normal / abnormal)

In [ ]:
import pandas as pd
from manifest import build_coughvid_manifest, build_icbhi_manifest, assign_splits

coughvid_df = build_coughvid_manifest()
coughvid_df = assign_splits(coughvid_df, train_frac=0.85)   # stratified by label
coughvid_df.head()

In [ ]:
icbhi_df = build_icbhi_manifest()
icbhi_df["split"] = "test"      # frozen external test set: never used for training/selection
icbhi_df.head()

## 2. Decode audio once to 16 kHz wav
COUGHVID mixes webm/ogg/wav; decoding webm every epoch would be very slow. This runs once (a few minutes).

In [ ]:
from cache_audio import cache_manifest_audio

combined = pd.concat([coughvid_df, icbhi_df], ignore_index=True, sort=False)
combined = cache_manifest_audio(combined)
combined["dataset"].value_counts()

## 3. Explore: class balance, durations, example spectrograms

In [ ]:
import matplotlib.pyplot as plt, numpy as np, soundfile as sf
from config import FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

cv = combined[combined.dataset == "coughvid"]
ic = combined[combined.dataset == "icbhi"]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
cv.groupby(["split", "label"]).size().unstack().plot.bar(ax=ax[0], title="COUGHVID (train/val)")
ic["label"].value_counts().plot.bar(ax=ax[1], title="ICBHI cycles (external test)")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=150); plt.show()

print("COUGHVID label balance:\n", cv["label"].value_counts(normalize=True).round(3))
print("\nICBHI label balance:\n", ic["label"].value_counts(normalize=True).round(3))
print("\nICBHI patients per label:\n", ic.groupby("label")["patient_id"].nunique())

In [ ]:
# clip durations (cached wavs are plain wav, so this is instant)
dur = cv["filepath"].apply(lambda p: sf.info(p).duration)
dur.hist(bins=40); plt.title("COUGHVID clip duration (s)"); plt.show()
print(dur.describe().round(2))
ic_len = ic["end_s"] - ic["start_s"]
print("\nICBHI cycle length (s):\n", ic_len.describe().round(2))

In [ ]:
import librosa.display
from preprocessing import preprocess_file
from spectrogram import audio_to_melspectrogram

samples = pd.concat([cv[cv.label == "normal"].sample(1, random_state=1),
                     cv[cv.label == "abnormal"].sample(1, random_state=1),
                     ic[ic.label == "normal"].sample(1, random_state=1),
                     ic[ic.label == "abnormal"].sample(1, random_state=1)])
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, (_, r) in zip(axes, samples.iterrows()):
    seg = (r.start_s, r.end_s) if pd.notna(r.get("start_s")) else None
    S = audio_to_melspectrogram(preprocess_file(r.filepath, segment=seg))
    ax.imshow(S, origin="lower", aspect="auto"); ax.set_title(f"{r.dataset} / {r.label}")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "example_spectrograms.png", dpi=150); plt.show()

## 4. Findings to note (fill in)
- COUGHVID normal/abnormal ratio: **TODO** → if very skewed, class weights in `train.py` matter
- ICBHI normal/abnormal ratio: **TODO** (ICBHI is mostly diseased patients — accuracy alone will mislead; use per-class recall)
- **Domain caveat:** COUGHVID = phone-recorded coughs; ICBHI = stethoscope lung sounds. The cross-dataset test measures how well a cough-trained model transfers to a very different signal — expect a large drop, and report it honestly rather than tuning on ICBHI.

## 5. Save the manifest

In [ ]:
from config import DATA_PROCESSED_DIR
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
combined.to_csv(DATA_PROCESSED_DIR / "manifest.csv", index=False)
print(combined.groupby(["dataset", "split", "label"]).size())

## 6. Quick smoke test (1 epoch), then the real training run

In [ ]:
!python src/model.py
!python src/train.py --epochs 1 --num-workers 2

In [ ]:
!python src/train.py --epochs 25 --num-workers 2

After training: `models/baseline_cnn_best.pt` and `training_history.json` are in `/kaggle/working/AeroSense/models/`.
Click **Save Version → Save & Run All** to keep them as notebook output, or download them from the Output tab.